# Figure 2
TC track density (a. obs, b. factual, c. factual-counterfactual in rows)

Import statements

In [1]:
import cartopy.crs as ccrs
from matplotlib import cm
import matplotlib.cbook as cbook
import matplotlib.colors as colors
import matplotlib as mpl
import matplotlib.pyplot as plt
import metpy.constants as mpconst
import numpy as np
import pandas as pd
import xarray
from scipy import stats

In [2]:
class MidpointNormalize(colors.Normalize):
    def __init__(self, vmin=None, vmax=None, vcenter=None, clip=False):
        self.vcenter = vcenter
        super().__init__(vmin, vmax, clip)

    def __call__(self, value, clip=None):
        # I'm ignoring masked values and all kinds of edge cases to make a
        # simple example...
        # Note also that we must extrapolate beyond vmin/vmax
        x, y = [self.vmin, self.vcenter, self.vmax], [0, 0.5, 1.]
        return np.ma.masked_array(np.interp(value, x, y,
                                            left=-np.inf, right=np.inf))

    def inverse(self, value):
        y, x = [self.vmin, self.vcenter, self.vmax], [0, 0.5, 1]
        return np.interp(value, x, y, left=-np.inf, right=np.inf)

In [3]:
# custom cmap
def truncate_colormap(cmap, minval=0.0, maxval=1.0, n=100):
    new_cmap = colors.LinearSegmentedColormap.from_list(
        'trunc({n},{a:.2f},{b:.2f})'.format(n=cmap.name, a=minval, b=maxval),
        cmap(np.linspace(minval, maxval, n)))
    return new_cmap

cmap = plt.get_cmap('viridis')
viridis_custom = truncate_colormap(cmap, 0, 0.9)

cmap = plt.get_cmap('gist_earth')
earth_custom = truncate_colormap(cmap, 0, 0.75)

custom = colors.LinearSegmentedColormap.from_list("", ["indigo","steelblue","greenyellow"], N=7)
custom_red = colors.LinearSegmentedColormap.from_list("", ["darkorange", "red"], N=3)

## Process density data

In [4]:
# function to analyze tracks
def read_track_data(filename):
    """
    Return all TC tracks in a given file
    """
    tracks = []
    current_track = []

    with open(filename, "r") as file:
        for line in file:
            if line.startswith("start"):
                if current_track:
                    tracks.append(current_track)
                    current_track = []
            else:
                parts = line.split()
                if len(parts) >= 5:
                    lon = float(parts[2])
                    if lon > 180:
                        lon -= 360
                    lat = float(parts[3])
                    current_track.append((lon, lat))

    if current_track:
        tracks.append(current_track)

    return tracks


def read_track_data_ibtracs(year):
    """
    read and return track data for a given year of IBTrACS data
    input:
        year (int): desired year
    output:
        tracks (list): list of tracks for that year
    notes: make sure the tracks output is the same as the tracks output for other "read_track_data" function
    """
    # open dataframe
    df = pd.read_csv(
        f"/glade/work/smhenry/NeuralGCM/data/ibtracs/IBTrACS_{year}_JASO.csv"
    )

    # filter out storms less than 54 hours
    df["time"] = pd.to_datetime(df["time"])
    lifetimes = df.groupby("stormid")["time"].agg(
        lambda x: (x.max() - x.min()).total_seconds() / 3600
    )
    min54h = lifetimes[lifetimes >= 54].index
    df_filtered = df[df["stormid"].isin(min54h)]
    df_filtered.loc[:, "lon"] = df_filtered["lon"].apply(
        lambda x: x - 360 if x > 180 else x
    )

    tracks = []

    # group by stormid
    for _, track_df in df_filtered.groupby("stormid"):
        # track_df = track_df.iloc[::2]
        track = list(zip(track_df["lon"], track_df["lat"]))
        tracks.append(track)

    tracks = tracks[::2]

    return tracks


# function to process density data
def get_density(tracks, save, load, savename=None):
    """
    Returns area-weighted 2d density smoothed with sliding window
    """

    # optional load from savename
    if load:
        density = xarray.open_dataset(
            "/glade/work/smhenry/NeuralGCM/data/density_06s/" + savename + ".nc"
        )
    else:
        # process tracks into lat and lon lists
        lons = []
        lats = []

        for i in range(len(tracks)):
            for j in range(len(tracks[i])):
                lons.append(tracks[i][j][0])
                lats.append(tracks[i][j][1])

        # create lat and lon bins and sliding window
        bin = 4  # dlat, dlon
        window = 10  # window dlat, dlon

        bin_lat = np.arange(0, 90 + bin, bin)  # -50, -50+dlat, ... 50-dlat, 50
        bin_lon = np.arange(-180, 180 + bin, bin)  # -180, -180+dlon, ... 180-dlon

        # get frequency from the smaller resolution
        frequency_nd, lon_edges, lat_edges = np.histogram2d(
            lons, lats, [bin_lon, bin_lat]
        )

        # compute center values of lat and lon bins
        bin_lon_centers = (lon_edges[:-1] + lon_edges[1:]) / 2
        bin_lat_centers = (lat_edges[:-1] + lat_edges[1:]) / 2

        # create frequency DataArray
        frequency = xarray.DataArray(
            frequency_nd.T,
            coords={"latitude": bin_lat[0:-1], "longitude": bin_lon[0:-1]},
            dims=["latitude", "longitude"],
            name="frequency",
        )

        # initalize density ndarray
        density_nd = np.zeros(np.shape(frequency_nd))

        # get frequency of sliding windows
        for i, lon in enumerate(bin_lon[0:-1]):
            for j, lat in enumerate(bin_lat[0:-1]):
                frequency_window = frequency.sel(
                    latitude=slice(lat, lat + window),
                    longitude=slice(lon, lon + window),
                ).sum(dim=["latitude", "longitude"])

                # calculate density = frequency / gridbox area
                R = mpconst.earth_avg_radius / 1000  # radius of earth, km
                A_window = (
                    np.pi
                    * R**2
                    * np.abs(np.sin(np.deg2rad(lat + window)) - np.sin(np.deg2rad(lat)))
                    * window
                    / 180
                )  # area of grid window, km2

                density_nd[i, j] = frequency_window / A_window  # season-1 km-2

        # create density DataArray
        density = xarray.DataArray(
            density_nd.T,
            coords={"latitude": bin_lat_centers, "longitude": bin_lon_centers},
            dims=["latitude", "longitude"],
            name="density [season-1 km-2]",
        )

        # optional save
        if save:
            density.to_netcdf(
                f"/glade/work/smhenry/NeuralGCM/data/density_06s/{savename}.nc"
            )

    # return density DataArray
    return density


# function to analyze genesis
def read_track_genesis_data(filename):
    """
    Return genesis data for all storms
    """
    tracks = []
    current_track = []

    with open(filename, "r") as file:
        for line in file:
            if line.startswith("start"):
                if current_track:
                    tracks.append(current_track)
                    current_track = []
            else:
                parts = line.split()
                if len(parts) >= 5:
                    lon = float(parts[2])
                    if lon > 180:
                        lon -= 360
                    lat = float(parts[3])
                    current_track.append((lon, lat))

    if current_track:
        tracks.append(current_track)

    return tracks


def read_track_genesis_data_ibtracs(year):
    """
    Return genesis data for all storms in ibtracs
    """
    # Open dataframe
    df = pd.read_csv(
        f"/glade/work/smhenry/NeuralGCM/data/ibtracs/IBTrACS_{year}_JASO.csv"
    )

    # filter out storms less than 54 hours
    df["time"] = pd.to_datetime(df["time"])
    lifetimes = df.groupby("stormid")["time"].agg(
        lambda x: (x.max() - x.min()).total_seconds() / 3600
    )
    min54h = lifetimes[lifetimes >= 54].index
    df_filtered = df[df["stormid"].isin(min54h)]
    df_filtered.loc[:, "lon"] = df_filtered["lon"].apply(
        lambda x: x - 360 if x > 180 else x
    )

    # get genesis points
    tracks = (
        df_filtered.groupby("stormid")
        .first()[["lon", "lat"]]
        .apply(tuple, axis=1)
        .tolist()
    )

    return tracks


# function to process genesis_density data
def get_genesis_density(tracks, save, load, savename=None):
    """
    Returns area-weighted 2d genesis_density smoothed with sliding window
    """

    # optional load from savename
    if load:
        genesis_density = xarray.open_dataset(
            "/glade/work/smhenry/NeuralGCM/data/genesis_density_06s/"
            + savename
            + ".nc"
        )
    else:
        # process tracks into lat and lon lists (only the first point of each track)
        lons = []
        lats = []

        for track in tracks:
            if track:  # ensure track is not empty
                lons.append(track[0][0])  # first longitude in track
                lats.append(track[0][1])  # first latitude in track

        # create lat and lon bins and sliding window
        bin = 4  # dlat, dlon
        window = 10  # window dlat, dlon

        bin_lat = np.arange(0, 90 + bin, bin)  # -50, -50+dlat, ... 50-dlat, 50
        bin_lon = np.arange(-180, 180 + bin, bin)  # -180, -180+dlon, ... 180-dlon

        # get frequency from the smaller resolution
        frequency_nd, lon_edges, lat_edges = np.histogram2d(
            lons, lats, [bin_lon, bin_lat]
        )

        # compute center values of lat and lon bins
        bin_lon_centers = (lon_edges[:-1] + lon_edges[1:]) / 2
        bin_lat_centers = (lat_edges[:-1] + lat_edges[1:]) / 2

        # create frequency DataArray
        frequency = xarray.DataArray(
            frequency_nd.T,
            coords={"latitude": bin_lat[0:-1], "longitude": bin_lon[0:-1]},
            dims=["latitude", "longitude"],
            name="frequency",
        )

        # initalize genesis_density ndarray
        genesis_density_nd = np.zeros(np.shape(frequency_nd))

        # get frequency of sliding windows
        for i, lon in enumerate(bin_lon[0:-1]):
            for j, lat in enumerate(bin_lat[0:-1]):
                frequency_window = frequency.sel(
                    latitude=slice(lat, lat + window),
                    longitude=slice(lon, lon + window),
                ).sum(dim=["latitude", "longitude"])

                # calculate genesis_density = frequency / gridbox area
                R = mpconst.earth_avg_radius / 1000  # radius of earth, km
                A_window = (
                    np.pi
                    * R**2
                    * np.abs(np.sin(np.deg2rad(lat + window)) - np.sin(np.deg2rad(lat)))
                    * window
                    / 180
                )  # area of grid window, km2

                genesis_density_nd[i, j] = frequency_window / A_window  # season-1 km-2

        # create genesis_density DataArray
        genesis_density = xarray.DataArray(
            genesis_density_nd.T,
            coords={"latitude": bin_lat_centers, "longitude": bin_lon_centers},
            dims=["latitude", "longitude"],
            name="density [season-1 km-2]",
        )

        # optional save
        if save:
            genesis_density.to_netcdf(
                f"/glade/work/smhenry/NeuralGCM/data/genesis_density_06s/{savename}.nc"
            )

    # return genesis_density DataArray
    return genesis_density

## Plotting function

In [5]:
# Build diverging colormap: blues/purples for neg, oranges/reds for pos
neg_cmap = plt.get_cmap("cool_r", 128)   # or whatever "custom" is
pos_cmap = plt.get_cmap("hot_r", 128)

neg_colors = neg_cmap(np.linspace(0, 1, 128))
pos_colors = pos_cmap(np.linspace(0, 1, 128))
all_colors = np.vstack([neg_colors, pos_colors])
diverging_cmap = colors.LinearSegmentedColormap.from_list("diverging_custom", all_colors)

In [6]:
def plot_density_r1(start, ax, title=None):
    """
    Plot year to year ensemble mean track density from a given file and optional save
    """

    nyr = 20

    density_yrlist = []

    # load year data
    for iyr in range(nyr):
        tracks = read_track_data_ibtracs(iyr + start)
        density = get_density(
            tracks,
            save=False,  # if load, save doesn't matter
            load=True,  # mean_density_NA_JASO_ERA5
            savename=f"density_{iyr+start}_JASO_IBTrACS",  # density_YYYY_JASO_IBTrACS
        )

        # take ensemble mean
        density_yrlist.append(density)

    # take year to year mean
    density_yr = xarray.concat(density_yrlist, dim="year")
    density_yrmean = density_yr.mean(dim="year")
    density_yrmean = density_yrmean["density"].rename("track density")
    density_yrmean = density_yrmean.assign_coords(
        longitude=(density_yrmean.longitude % 360)
    ).sortby("longitude")

    # set up plot
    ax.set_extent([100, 350, 1.4, 50], crs=ccrs.PlateCarree())
    ax.coastlines(resolution="10m", linewidth=0.8, color="gray")

    gl = ax.gridlines(
        draw_labels=True, linewidth=0.3, color="gray", alpha=0.7, linestyle="--"
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 10}
    gl.ylabel_style = {"size": 10}

    # plot density
    # density_yrmean["track density"].plot.contourf(
    density_yrmean.plot.contourf(
        x="longitude",
        y="latitude",
        robust=True,
        ax=ax,
        transform=ccrs.PlateCarree(),
        extend="max",
        cmap="jet",
        levels=np.linspace(1e-5, 8e-5, 11),
    )

    if title:
        ax.set_title(title, fontsize=12)


def plot_density_r2(filenames, start, c_or_f, ax, exclude=None, title=None):
    """
    Plot year to year ensemble mean track density from a given file and optional save
    """

    nyr = np.shape(filenames)[0]
    nens = np.shape(filenames)[1]

    density_yrlist = []

    # load year data
    for iyr in range(nyr):
        # load ensemble member density data
        density_enslist = []
        for iens in range(nens):
            if exclude:
                is_included = True
                for yr_ens in exclude:
                    if (iyr + start == yr_ens[0]) and iens == (yr_ens[1] - 1):
                        is_included = False
                if is_included:
                    tracks = read_track_data(filenames[iyr][iens])
                    density = get_density(
                        tracks,
                        save=False,  # if load, save doesn't matter
                        load=True,  # mean_density_NA_JASO_factual
                        savename=f"density_{iyr+start}_ens{iens+1}_JASO_{c_or_f}",  # density_YYYY_ensXX_JASO_(counter)factual
                    )
                    density_enslist.append(density)
            else:
                tracks = read_track_data(filenames[iyr][iens])
                density = get_density(
                    tracks,
                    save=False,  # if load, save doesn't matter
                    load=True,  # mean_density_NA_JASO_factual
                    savename=f"density_{iyr+start}_ens{iens+1}_JASO_{c_or_f}",  # density_YYYY_ensXX_JASO_(counter)factual
                )
                density_enslist.append(density)
        density_xr = xarray.concat(density_enslist, dim="ens")

        # take ensemble mean
        density_ensmean = density_xr.mean(dim="ens")
        density_yrlist.append(density_ensmean)

    # take year to year mean
    density_yr = xarray.concat(density_yrlist, dim="year")
    density_yrmean = density_yr.mean(dim="year")
    density_yrmean = density_yrmean["density"].rename("track density")
    # density_yrmean = density_yrmean.rename("track density")
    density_yrmean = density_yrmean.assign_coords(
        longitude=(density_yrmean.longitude % 360)
    ).sortby("longitude")

    # set up plot
    ax.set_extent([100, 350, 1.4, 50], crs=ccrs.PlateCarree())
    ax.coastlines(resolution="10m", linewidth=0.8, color="gray")

    gl = ax.gridlines(
        draw_labels=True, linewidth=0.3, color="gray", alpha=0.7, linestyle="--"
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 10}
    gl.ylabel_style = {"size": 10}

    # plot density
    density_yrmean.plot.contourf(
        x="longitude",
        y="latitude",
        robust=True,
        ax=ax,
        transform=ccrs.PlateCarree(),
        extend="max",
        cmap="jet",
        levels=np.linspace(1e-5, 8e-5, 11),
    )

    if title:
        ax.set_title(title, fontsize=12)


def plot_density_r3(
    filenames_f, filenames_cf, start, ax, exclude_f=None, exclude_cf=None, title=None
):
    """
    Plot year to year ensemble mean track density from a given file and optional save
    """
    nyr = 20
    nens = 20

    density_yrlist_f = []

    # load year data
    for iyr in range(nyr):
        # load ensemble member density data
        density_enslist_f = []
        for iens in range(nens):
            if exclude_f:
                is_included = True
                for yr_ens in exclude_f:
                    if (iyr + start == yr_ens[0]) and iens == (yr_ens[1] - 1):
                        is_included = False
                if is_included:
                    tracks = read_track_data(filenames_f[iyr][iens])
                    density_f = get_density(
                        tracks,
                        save=False,  # if load, save doesn't matter
                        load=True,  # mean_density_NA_JASO_factual
                        savename=f"density_{iyr+start}_ens{iens+1}_JASO_factual",  # density_YYYY_ensXX_JASO_(counter)factual
                    )
                    density_enslist_f.append(density_f)
            else:
                tracks = read_track_data(filenames_f[iyr][iens])
                density_f = get_density(
                    tracks,
                    save=False,  # if load, save doesn't matter
                    load=True,  # mean_density_NA_JASO_factual
                    savename=f"density_{iyr+start}_ens{iens+1}_JASO_factual",  # density_YYYY_ensXX_JASO_(counter)factual
                )
                density_enslist_f.append(density_f)

        density_xr_f = xarray.concat(density_enslist_f, dim="ens")

        # take ensemble mean
        density_ensmean_f = density_xr_f.mean(dim="ens")
        density_yrlist_f.append(density_ensmean_f)

    density_yr_f = xarray.concat(density_yrlist_f, dim="year")
    density_yr_f = density_yr_f["density"].rename("track density")
    density_yr_f = density_yr_f.assign_coords(
        longitude=(density_yr_f.longitude % 360)
    ).sortby("longitude")

    density_yrlist_cf = []

    # load year data
    for iyr in range(nyr):
        # load ensemble member density data
        density_enslist_cf = []
        for iens in range(nens):
            if exclude_cf:
                is_included = True
                for yr_ens in exclude_cf:
                    if (iyr + start == yr_ens[0]) and iens == (yr_ens[1] - 1):
                        is_included = False
                if is_included:
                    tracks = read_track_data(filenames_cf[iyr][iens])
                    density_cf = get_density(
                        tracks,
                        save=False,  # if load, save doesn't matter
                        load=True,  # mean_density_NA_JASO_cfactual
                        savename=f"density_{iyr+start}_ens{iens+1}_JASO_counterfactual",  # density_YYYY_ensXX_JASO_(counter)factual
                    )
                    density_enslist_cf.append(density_cf)
            else:
                tracks = read_track_data(filenames_cf[iyr][iens])
                density_cf = get_density(
                    tracks,
                    save=False,  # if load, save doesn't matter
                    load=True,  # mean_density_NA_JASO_cfactual
                    savename=f"density_{iyr+start}_ens{iens+1}_JASO_counterfactual",  # density_YYYY_ensXX_JASO_(counter)factual
                )
                density_enslist_cf.append(density_cf)

        density_xr_cf = xarray.concat(density_enslist_cf, dim="ens")

        # take ensemble mean
        density_ensmean_cf = density_xr_cf.mean(dim="ens")
        density_yrlist_cf.append(density_ensmean_cf)

    density_yr_cf = xarray.concat(density_yrlist_cf, dim="year")
    density_yr_cf = density_yr_cf["density"].rename("track density")
    density_yr_cf = density_yr_cf.assign_coords(
        longitude=(density_yr_cf.longitude % 360)
    ).sortby("longitude")

    density_yrmean_f = density_yr_f.mean(dim="year")
    density_yrmean_cf = density_yr_cf.mean(dim="year")

    density_yrmean_f.close()
    density_yrmean_cf.close()
    diff = density_yrmean_f - density_yrmean_cf

    # significance
    t_stat, p_val = stats.ttest_ind(
        density_yr_f,
        density_yr_cf,
        axis=0,
        equal_var=False,
    )

    # mask
    alpha = 0.01
    significant_mask = p_val < alpha

    # set up plot
    ax.set_extent([100, 350, 1.4, 50], crs=ccrs.PlateCarree())
    ax.coastlines(resolution="10m", linewidth=0.8, color="gray")

    gl = ax.gridlines(
        draw_labels=True, linewidth=0.3, color="gray", alpha=0.7, linestyle="--"
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 10}
    gl.ylabel_style = {"size": 10}

    # plot density
    levels = np.linspace(-8e-5, -1e-5, 11)

    # --- build diverging colormap ---
    neg_cmap = custom          # your existing colormap for negatives
    pos_cmap = custom_red   # oranges/reds for positives

    neg_colors = neg_cmap(np.linspace(0, 1, 256))
    pos_colors = pos_cmap(np.linspace(0, 1, 256))
    all_colors = np.vstack([neg_colors, pos_colors])
    diverging_cmap = colors.LinearSegmentedColormap.from_list("div_custom", all_colors)

    # --- mask to significant values only ---
    diff_significant = diff.where(significant_mask)

    # --- symmetric levels for diverging colorbar ---
    levels = np.arange(-8e-5, 3e-5, 1e-5)
    norm = colors.TwoSlopeNorm(vmin=-8e-5, vcenter=0, vmax=4e-5)

    diff_significant.plot.contourf(
        x="longitude",
        y="latitude",
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=diverging_cmap,
        norm=norm,
        cbar_kwargs={"label": "genesis density"},
        levels=levels,
        extend="both",
    )

    if title:
        ax.set_title(title, fontsize=12)
    
    # masked_diff = diff.where((diff >= 0.5e-5)|(diff <= -0.5e-5))
    
    # masked_diff.plot.contourf(
    #     x="longitude",
    #     y="latitude",
    #     ax=ax,
    #     transform=ccrs.PlateCarree(),
    #     cmap=custom,
    #      norm=colors.PowerNorm(gamma=0.8, vmin=levels[0], vmax=levels[-1]),
    #     levels=levels,
    #     extend="min",
    # )

    # significant_mask = np.where(diff != 0, p_val < alpha, False)

    # # Plot stippling for significance
    # lon, lat = np.meshgrid(diff.longitude, diff.latitude)
    # ax.scatter(
    #     lon[significant_mask],
    #     lat[significant_mask],
    #     color="black",
    #     s=1,
    #     # alpha=0.5,
    #     transform=ccrs.PlateCarree(),
    #     marker=".",
    # )

    # # # plot significance contours
    # # ax.contour(
    # #     diff["longitude"],
    # #     diff["latitude"],
    # #     p_val,
    # #     # robust=True,
    # #     transform=ccrs.PlateCarree(),
    # #     levels=[0, alpha],
    # #     colors="k",
    # #     linestyles="dashed",
    # #     linewidths=1.5,
    # # )

    # if title:
    #     ax.set_title(title, fontsize=12)

In [7]:
def plot_genesis_density_r1(start, ax, title=None):
    """
    Plot year to year ensemble mean track genesis_density from a given file and optional save
    """

    nyr = 20

    genesis_density_yrlist = []

    # load year data
    for iyr in range(nyr):
        tracks = read_track_data_ibtracs(iyr + start)
        genesis_density = get_genesis_density(
            tracks,
            save=False,  # if load, save doesn't matter
            load=True,  # mean_genesis_density_NA_JASO_ERA5
            savename=f"genesis_density_{iyr+start}_JASO_IBTrACS",  # genesis_density_YYYY_JASO_IBTrACS
        )

        # take ensemble mean
        genesis_density_yrlist.append(genesis_density)

    # take year to year mean
    genesis_density_yr = xarray.concat(genesis_density_yrlist, dim="year")
    genesis_density_yrmean = genesis_density_yr.mean(dim="year")
    genesis_density_yrmean = genesis_density_yrmean["density"].rename("genesis density")
    genesis_density_yrmean = genesis_density_yrmean.assign_coords(
        longitude=(genesis_density_yrmean.longitude % 360)
    ).sortby("longitude")

    # set up plot
    ax.set_extent([100, 350, 1.4, 50], crs=ccrs.PlateCarree())
    ax.coastlines(resolution="10m", linewidth=0.8, color="gray")

    gl = ax.gridlines(
        draw_labels=True, linewidth=0.3, color="gray", alpha=0.7, linestyle="--"
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 10}
    gl.ylabel_style = {"size": 10}

    # plot genesis_density
    genesis_density_yrmean.plot.contourf(
        x="longitude",
        y="latitude",
        robust=True,
        ax=ax,
        transform=ccrs.PlateCarree(),
        extend="max",
        cmap="jet",
        levels=np.linspace(1e-6, 5e-6, 11),
    )

    if title:
        ax.set_title(title, fontsize=14)


def plot_genesis_density_r2(filenames, start, c_or_f, ax, exclude=None, title=None):
    """
    Plot year to year ensemble mean track genesis_density from a given file and optional save
    """

    nyr = np.shape(filenames)[0]
    nens = np.shape(filenames)[1]

    genesis_density_yrlist = []

    # load year data
    for iyr in range(nyr):
        # load ensemble member genesis_density data
        genesis_density_enslist = []
        for iens in range(nens):
            if exclude:
                is_included = True
                for yr_ens in exclude:
                    if (iyr + start == yr_ens[0]) and iens == (yr_ens[1] - 1):
                        is_included = False
                if is_included:
                    tracks = read_track_data(filenames[iyr][iens])
                    genesis_density = get_genesis_density(
                        tracks,
                        save=False,  # if load, save doesn't matter
                        load=True,  # mean_genesis_density_NA_JASO_factual
                        savename=f"genesis_density_{iyr+start}_ens{iens+1}_JASO_{c_or_f}",  # genesis_density_YYYY_ensXX_JASO_(counter)factual
                    )
                    genesis_density_enslist.append(genesis_density)
            else:
                tracks = read_track_data(filenames[iyr][iens])
                genesis_density = get_genesis_density(
                    tracks,
                    save=False,  # if load, save doesn't matter
                    load=True,  # mean_genesis_density_NA_JASO_factual
                    savename=f"genesis_density_{iyr+start}_ens{iens+1}_JASO_{c_or_f}",  # genesis_density_YYYY_ensXX_JASO_(counter)factual
                )
                genesis_density_enslist.append(genesis_density)
        genesis_density_xr = xarray.concat(genesis_density_enslist, dim="ens")

        # take ensemble mean
        genesis_density_ensmean = genesis_density_xr.mean(dim="ens")
        genesis_density_yrlist.append(genesis_density_ensmean)

    # take year to year mean
    genesis_density_yr = xarray.concat(genesis_density_yrlist, dim="year")
    genesis_density_yrmean = genesis_density_yr.mean(dim="year")
    genesis_density_yrmean = genesis_density_yrmean["density"].rename("genesis density")
    genesis_density_yrmean = genesis_density_yrmean.assign_coords(
        longitude=(genesis_density_yrmean.longitude % 360)
    ).sortby("longitude")

    # set up plot
    ax.set_extent([100, 350, 1.4, 50], crs=ccrs.PlateCarree())
    ax.coastlines(resolution="10m", linewidth=0.8, color="gray")

    gl = ax.gridlines(
        draw_labels=True, linewidth=0.3, color="gray", alpha=0.7, linestyle="--"
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 10}
    gl.ylabel_style = {"size": 10}

    # plot genesis_density

    genesis_density_yrmean.plot.contourf(
        x="longitude",
        y="latitude",
        robust=True,
        ax=ax,
        transform=ccrs.PlateCarree(),
        extend="max",
        cmap="jet",
        levels=np.linspace(1e-6, 5e-6, 11),
    )

    if title:
        ax.set_title(title, fontsize=14)


def plot_genesis_density_r3(
    filenames_f, filenames_cf, start, ax, exclude_f=None, exclude_cf=None, title=None
):
    """
    Plot year to year ensemble mean track genesis_density from a given file and optional save
    """
    nyr = 20
    nens = 20

    genesis_density_yrlist_f = []

    # load year data
    for iyr in range(nyr):
        # load ensemble member genesis_density data
        genesis_density_enslist_f = []
        for iens in range(nens):
            if exclude_f:
                is_included = True
                for yr_ens in exclude_f:
                    if (iyr + start == yr_ens[0]) and iens == (yr_ens[1] - 1):
                        is_included = False
                if is_included:
                    tracks = read_track_data(filenames_f[iyr][iens])
                    genesis_density_f = get_genesis_density(
                        tracks,
                        save=False,  # if load, save doesn't matter
                        load=True,  # mean_genesis_density_NA_JASO_factual
                        savename=f"genesis_density_{iyr+start}_ens{iens+1}_JASO_factual",  # genesis_density_YYYY_ensXX_JASO_(counter)factual
                    )
                    genesis_density_enslist_f.append(genesis_density_f)
            else:
                tracks = read_track_data(filenames_f[iyr][iens])
                genesis_density_f = get_genesis_density(
                    tracks,
                    save=False,  # if load, save doesn't matter
                    load=True,  # mean_genesis_density_NA_JASO_factual
                    savename=f"genesis_density_{iyr+start}_ens{iens+1}_JASO_factual",  # genesis_density_YYYY_ensXX_JASO_(counter)factual
                )
                genesis_density_enslist_f.append(genesis_density_f)

        genesis_density_xr_f = xarray.concat(genesis_density_enslist_f, dim="ens")

        # take ensemble mean
        genesis_density_ensmean_f = genesis_density_xr_f.mean(dim="ens")
        genesis_density_yrlist_f.append(genesis_density_ensmean_f)

    genesis_density_yr_f = xarray.concat(genesis_density_yrlist_f, dim="year")
    genesis_density_yr_f = genesis_density_yr_f['density'].rename("genesis_density")
    genesis_density_yr_f = genesis_density_yr_f.assign_coords(
        longitude=(genesis_density_yr_f.longitude % 360)
    ).sortby("longitude")

    genesis_density_yrlist_cf = []

    # load year data
    for iyr in range(nyr):
        # load ensemble member genesis_density data
        genesis_density_enslist_cf = []
        for iens in range(nens):
            if exclude_cf:
                is_included = True
                for yr_ens in exclude_cf:
                    if (iyr + start == yr_ens[0]) and iens == (yr_ens[1] - 1):
                        is_included = False
                if is_included:
                    tracks = read_track_data(filenames_cf[iyr][iens])
                    genesis_density_cf = get_genesis_density(
                        tracks,
                        save=False,  # if load, save doesn't matter
                        load=True,  # mean_genesis_density_NA_JASO_cfactual
                        savename=f"genesis_density_{iyr+start}_ens{iens+1}_JASO_counterfactual",  # genesis_density_YYYY_ensXX_JASO_(counter)factual
                    )
                    genesis_density_enslist_cf.append(genesis_density_cf)
            else:
                tracks = read_track_data(filenames_cf[iyr][iens])
                genesis_density_cf = get_genesis_density(
                    tracks,
                    save=False,  # if load, save doesn't matter
                    load=True,  # mean_genesis_density_NA_JASO_cfactual
                    savename=f"genesis_density_{iyr+start}_ens{iens+1}_JASO_counterfactual",  # genesis_density_YYYY_ensXX_JASO_(counter)factual
                )
                genesis_density_enslist_cf.append(genesis_density_cf)

        genesis_density_xr_cf = xarray.concat(genesis_density_enslist_cf, dim="ens")

        # take ensemble mean
        genesis_density_ensmean_cf = genesis_density_xr_cf.mean(dim="ens")
        genesis_density_yrlist_cf.append(genesis_density_ensmean_cf)

    genesis_density_yr_cf = xarray.concat(genesis_density_yrlist_cf, dim="year")
    genesis_density_yr_cf = genesis_density_yr_cf['density'].rename("genesis density")
    genesis_density_yr_cf = genesis_density_yr_cf.assign_coords(
        longitude=(genesis_density_yr_cf.longitude % 360)
    ).sortby("longitude")

    genesis_density_yrmean_f = genesis_density_yr_f.mean(dim="year")
    genesis_density_yrmean_cf = genesis_density_yr_cf.mean(dim="year")

    genesis_density_yrmean_f.close()
    genesis_density_yrmean_cf.close()
    diff = genesis_density_yrmean_f - genesis_density_yrmean_cf
    diff.rename("genesis_density")

    # significance
    t_stat, p_val = stats.ttest_ind(
        genesis_density_yr_f,
        genesis_density_yr_cf,
        axis=0,
        equal_var=False,
    )

    # mask
    alpha = 0.01
    significant_mask = p_val < alpha

    # set up plot
    ax.set_extent([100, 350, 1.4, 50], crs=ccrs.PlateCarree())
    ax.coastlines(resolution="10m", linewidth=0.8, color="gray")

    gl = ax.gridlines(
        draw_labels=True, linewidth=0.3, color="gray", alpha=0.7, linestyle="--"
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 10}
    gl.ylabel_style = {"size": 10}

    # plot genesis_density
    levels = np.linspace(-6e-6, -1e-6, 11)

    # --- build diverging colormap ---
    neg_cmap = custom          # your existing colormap for negatives
    pos_cmap = custom_red   # oranges/reds for positives

    neg_colors = neg_cmap(np.linspace(0, 1, 256))
    pos_colors = pos_cmap(np.linspace(0, 1, 256))
    all_colors = np.vstack([neg_colors, pos_colors])
    diverging_cmap = colors.LinearSegmentedColormap.from_list("div_custom", all_colors)

    # --- mask to significant values only ---
    diff_significant = diff.where(significant_mask)

    # --- symmetric levels for diverging colorbar ---
    levels = np.arange(-6e-6, 3e-6, 1e-6)
    norm = colors.TwoSlopeNorm(vmin=-6e-6, vcenter=0, vmax=3e-6)
    
    diff_significant.plot.contourf(
        x="longitude",
        y="latitude",
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=diverging_cmap,
        norm=norm,
        cbar_kwargs={"label": "genesis density"},
        levels=levels,
        extend="both",
    )

    if title:
        ax.set_title(title, fontsize=14)

        
    # masked_diff = diff.where((diff >= 0.5e-6)|(diff <= -0.5e-6))
    
    # masked_diff.plot.contourf(
    #     x="longitude",
    #     y="latitude",
    #     ax=ax,
    #     transform=ccrs.PlateCarree(),
    #     cmap=custom,
    #      norm=colors.PowerNorm(gamma=0.8, vmin=levels[0], vmax=levels[-1]),
    #     cbar_kwargs={"label":"genesis density",},
    #     levels=levels,
    #     extend="min",
    # )

    # significant_mask = np.where(diff != 0, p_val < alpha, False)

    # # Plot stippling for significance
    # lon, lat = np.meshgrid(diff.longitude, diff.latitude)
    # ax.scatter(
    #     lon[significant_mask],
    #     lat[significant_mask],
    #     color="black",
    #     s=1,
    #     # alpha=0.5,
    #     transform=ccrs.PlateCarree(),
    #     marker=".",
    # )

    # # # plot significance contours
    # # ax.contour(
    # #     diff["longitude"],
    # #     diff["latitude"],
    # #     p_val,
    # #     # robust=True,
    # #     transform=ccrs.PlateCarree(),
    # #     levels=[0, alpha],
    # #     colors="k",
    # #     linestyles="dashed",
    # #     linewidths=1.5,
    # # )

    # if title:
    #     ax.set_title(title, fontsize=14)

## Plot

In [ ]:
nens = 20
years = np.arange(2004, 2023 + 1, 1)

names_counterfactual = [
    [
        f"/glade/work/smhenry/NeuralGCM/data/tracks/counterfactual/ens{X}_{YYYY}_JASO_TC_tracks_counterfactual.txt"
        for X in range(1, nens + 1)
    ]
    for YYYY in years
]

names_factual = [
    [
        f"/glade/work/smhenry/NeuralGCM/data/tracks/factual/ens{X}_{YYYY}_JASO_TC_tracks_factual.txt"
        for X in range(1, nens + 1)
    ]
    for YYYY in years
]

# names_ERA5 = [
#     f"/glade/work/smhenry/NeuralGCM/data/tracks/ERA5/ERA5_{YYYY}_JASO_TC_tracks_vorticity.txt"
#     for YYYY in years
# ]

f_exclude = [[2005, 17], [2012, 14], [2017, 3], [2017, 12], [2017, 13], [2021, 16]]
cf_exclude = [[2009, 12], [2017, 15]]

# set up figure 1 col 3 row

fig, axs = plt.subplots(
    ncols=2,
    nrows=3,
    figsize=(12, 5),
    subplot_kw={"projection": ccrs.PlateCarree(central_longitude=180)},
)
axs = axs.flatten()

# plot ERA5 obs on axs[0]
plot_density_r1(
    years[0],
    axs[1],
    title="(b) IBTrACS TDF",
)

# plot factual on axs[1]
plot_density_r2(
    names_factual,
    years[0],
    "factual",
    axs[3],
    exclude=f_exclude,
    title="(d) Factual TDF",
)

# plot factual-counterfactual on axs[2]
plot_density_r3(
    names_factual,
    names_counterfactual,
    years[0],
    axs[5],
    exclude_f=f_exclude,
    exclude_cf=cf_exclude,
    title="(f) (Factual - Counterfactual) TDF",
)

# plot ERA5 obs on axs[0]
plot_genesis_density_r1(
    years[0],
    axs[0],
    title="(a) IBTrACS GDF",
)

# plot factual on axs[1]
plot_genesis_density_r2(
    names_factual,
    years[0],
    "factual",
    axs[2],
    exclude=f_exclude,
    title="(c) Factual GDF",
)

# plot factual-counterfactual on axs[2]
plot_genesis_density_r3(
    names_factual,
    names_counterfactual,
    years[0],
    axs[4],
    exclude_f=f_exclude,
    exclude_cf=cf_exclude,
    title="(e) (Factual - Counterfactual) GDF",
)

# plt.suptitle("2004-2023 Mean TC Track and Genesis Density (season-1 km-2)", fontsize=12)
plt.savefig("./figs/figure2.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
nens = 20
years = np.arange(2004, 2023 + 1, 1)

names_counterfactual = [
    [
        f"/glade/work/smhenry/NeuralGCM/data/tracks/counterfactual/ens{X}_{YYYY}_JASO_TC_tracks_counterfactual.txt"
        for X in range(1, nens + 1)
    ]
    for YYYY in years
]

names_factual = [
    [
        f"/glade/work/smhenry/NeuralGCM/data/tracks/factual/ens{X}_{YYYY}_JASO_TC_tracks_factual.txt"
        for X in range(1, nens + 1)
    ]
    for YYYY in years
]

# names_ERA5 = [
#     f"/glade/work/smhenry/NeuralGCM/data/tracks/ERA5/ERA5_{YYYY}_JASO_TC_tracks_vorticity.txt"
#     for YYYY in years
# ]

f_exclude = [[2005, 17], [2012, 14], [2017, 3], [2017, 12], [2017, 13], [2021, 16]]
cf_exclude = [[2009, 12], [2017, 15]]

# set up figure 1 col 3 row

fig, axs = plt.subplots(
    ncols=2,
    nrows=4,
    figsize=(12, 7),
    subplot_kw={"projection": ccrs.PlateCarree(central_longitude=180)},
)
axs = axs.flatten()

# plot ERA5 obs on axs[0]
plot_density_r1(
    years[0],
    axs[1],
    title="(b) IBTrACS TDF",
)

# plot factual on axs[1]
plot_density_r2(
    names_factual,
    years[0],
    "factual",
    axs[3],
    exclude=f_exclude,
    title="(d) Factual TDF",
)

# plot cf on axs[1]
plot_density_r2(
    names_counterfactual,
    years[0],
    "counterfactual",
    axs[5],
    exclude=cf_exclude,
    title="(f) Counterfactual TDF",
)

# plot factual-counterfactual on axs[2]
plot_density_r3(
    names_factual,
    names_counterfactual,
    years[0],
    axs[7],
    exclude_f=f_exclude,
    exclude_cf=cf_exclude,
    title="(h) (Factual - Counterfactual) TDF",
)

# plot ERA5 obs on axs[0]
plot_genesis_density_r1(
    years[0],
    axs[0],
    title="(a) IBTrACS GDF",
)

# plot factual on axs[1]
plot_genesis_density_r2(
    names_factual,
    years[0],
    "factual",
    axs[2],
    exclude=f_exclude,
    title="(c) Factual GDF",
)

# plot cf on axs[1]
plot_genesis_density_r2(
    names_counterfactual,
    years[0],
    "counterfactual",
    axs[4],
    exclude=cf_exclude,
    title="(e) Counterfactual GDF",
)

# plot factual-counterfactual on axs[2]
plot_genesis_density_r3(
    names_factual,
    names_counterfactual,
    years[0],
    axs[6],
    exclude_f=f_exclude,
    exclude_cf=cf_exclude,
    title="(g) (Factual - Counterfactual) GDF",
)

# plt.suptitle("2004-2023 Mean TC Track and Genesis Density (season-1 km-2)", fontsize=12)
plt.savefig("./figs/figure2_with-cf.png", dpi=600, bbox_inches="tight")
plt.savefig("./figs/figure2_with-cf.pdf", dpi=600, bbox_inches="tight")
plt.show()